# 🏷️ Sınıf Bazlı Otomatik Etiketleme & YOLO Veri Seti Hazırlama (Pipeline)
Bu notebook, her kategoriyi (**bag, belt, gloves, necklace, scarf, tie, watch, sock**) **ayrı ayrı hücrelerde** bağımsız olarak etiketlemenizi, etiketlenen tüm sınıfları tek bir YOLO veri setinde toplayıp **Train/Val (%80 - %20) split** yaparak `data.yaml` ile birlikte eğitime hazır hale getirmenizi sağlar.

### 🚀 Mimarinin Özellikleri:
1. **Çift Dedektör (Grounding DINO + YOLO-World):** Biri kaçırsa diğeri yakalayarak maksimum nesne tespiti sağlar.
2. **Google SigLIP Doğrulayıcı:** Yanlış tespitleri (arka plan, kıyafet, ten vb.) otomatik eler.
3. **IoU Süzgeci:** Üst üste binen mükerrer etiketleri siler.
4. **Resume (Kaldığı Yerden Devam) Desteği:** Zaten `.txt` etiketi oluşmuş resimleri atlayarak zamandan tasarruf eder.
5. **Otomatik Dataset Birleştirme & YAML:** Tüm sınıfları tek tıkla YOLOv8/YOLOv11 formatında toplar.

### 🟢 ADIM 1: Google Drive'ı Bağlama

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

### 🟢 ADIM 2: Gerekli Kütüphanelerin Kurulumu

In [ ]:
# Tüm bağımlılıkları çakışmasız ve uyumlu en güncel sürümleriyle tek seferde yükseltip kuruyoruz
!pip install -q -U transformers tokenizers huggingface_hub
!pip install -q -U roboflow ultralytics supervision autodistill autodistill-grounding-dino

# Kurulum kontrolü
import transformers, supervision, ultralytics, autodistill
print('✅ Tüm kütüphaneler çakışmasız ve başarıyla kuruldu!')


### 🟢 ADIM 3: Ön İşleme - Bozuk Resimleri Temizleme (Opsiyonel)

In [ ]:
import os
from PIL import Image
import cv2

BASE_DIR = "/content/gdrive/MyDrive/kiyafetler"
CATEGORIES = ["bag", "belt", "gloves", "necklace", "scarf", "tie", "watch", "sock"]

def clean_corrupted_images(folder_path):
    deleted_count = 0
    valid_count = 0
    if not os.path.exists(folder_path):
        return
    print(f"🔍 Taranıyor: {os.path.basename(folder_path)} ...")
    for fname in os.listdir(folder_path):
        fpath = os.path.join(folder_path, fname)
        if not fname.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp')):
            continue
        is_corrupt = False
        if os.path.getsize(fpath) < 100:
            is_corrupt = True
        else:
            try:
                with Image.open(fpath) as img:
                    img.verify()
                cv_img = cv2.imread(fpath)
                if cv_img is None or cv_img.size == 0:
                    is_corrupt = True
            except Exception:
                is_corrupt = True
        if is_corrupt:
            try:
                os.remove(fpath)
                deleted_count += 1
            except Exception:
                pass
        else:
            valid_count += 1
    print(f"✅ {os.path.basename(folder_path)}: {valid_count} sağlam resim, {deleted_count} bozuk resim silindi.\n")

for cat in CATEGORIES:
    cat_path = os.path.join(BASE_DIR, cat)
    if not os.path.exists(cat_path):
        cat_path = os.path.join("/content/gdrive/MyDrive", cat)
    clean_corrupted_images(cat_path)

### 🟢 ADIM 3.1: Resimleri Standart İsimlendirme (Ön İşleme - Opsiyonel)
Her klasördeki resimleri `bag_0001.jpg`, `belt_0001.jpg` gibi standart ve çakışmasız formatta yeniden adlandırır (Türkçe karakterleri ve boşlukları temizler).

In [ ]:
import os

BASE_DIR = "/content/gdrive/MyDrive/kiyafetler"
CATEGORIES = ["bag", "belt", "gloves", "necklace", "scarf", "tie", "watch", "sock"]

def standardize_image_names(folder_path, prefix):
    if not os.path.exists(folder_path):
        return
    files = sorted([f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp', '.bmp'))])
    print(f"🏷️ {prefix.upper()}: {len(files)} görsel yeniden adlandırılıyor...")
    for idx, fname in enumerate(files, start=1):
        ext = os.path.splitext(fname)[1].lower()
        new_name = f"{prefix}_{idx:04d}{ext}"
        old_p = os.path.join(folder_path, fname)
        new_p = os.path.join(folder_path, new_name)
        if old_p != new_p:
            os.rename(old_p, new_p)
    print(f"✅ {prefix.upper()} klasörü tamamlandı! (Örnek: {prefix}_0001.jpg)\n")

# Çalıştırmak için:
for cat in CATEGORIES:
    cat_p = os.path.join(BASE_DIR, cat)
    if not os.path.exists(cat_p):
        cat_p = os.path.join("/content/gdrive/MyDrive", cat)
    standardize_image_names(cat_p, cat)


### 🟢 ADIM 4: Ortak Çoklu Modelli Etiketleme Motoru
*Bu hücreyi 1 kez çalıştırın. Ardından aşağıdan dilediğiniz sınıf hücresini bağımsızca çalıştırabilirsiniz.*

In [ ]:
# --- BERT UYUMLULUK YAMASI ---
import transformers
if not hasattr(transformers.BertModel, "get_head_mask"):
    def dummy_get_head_mask(self, head_mask, num_hidden_layers, is_attention_chunked=False):
        return [None] * num_hidden_layers
    transformers.BertModel.get_head_mask = dummy_get_head_mask

import os, glob, cv2, torch, numpy as np
from PIL import Image
from transformers import AutoProcessor, AutoModelForZeroShotImageClassification
from ultralytics import YOLOWorld
from autodistill_grounding_dino import GroundingDINO
from autodistill.detection import CaptionOntology
import supervision as sv

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Çalışma Ortamı: {device.upper()}")

# SigLIP Doğrulayıcı Modeli Yükleme
print("📥 Google SigLIP Doğrulayıcı Modeli Yükleniyor...")
siglip_name = "google/siglip-base-patch16-224"
siglip_processor = AutoProcessor.from_pretrained(siglip_name)
siglip_model = AutoModelForZeroShotImageClassification.from_pretrained(siglip_name).to(device)
print("✅ SigLIP Modeli Hazır!")

box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()
BASE_DIR = "/content/gdrive/MyDrive/kiyafetler"

def verify_crop(crop_img, target_label, min_score=0.20):
    """SigLIP ile kırpılan nesnenin doğruluğunu kontrol eder (Optimize Sigmoid)"""
    candidate_labels = [f"a photo of a {target_label}", "a photo of empty background or blurry texture"]
    inputs = siglip_processor(text=candidate_labels, images=crop_img, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        logits = siglip_model(**inputs).logits_per_image
        probs = logits.sigmoid()[0]
    target_score = probs[0].item()
    bg_score = probs[1].item()
    is_valid = (target_score >= min_score) or (target_score > bg_score)
    return is_valid, target_score

def run_auto_labeling(category_name, class_id, dino_prompts, yolo_prompts, siglip_candidates=None, box_threshold=0.20, yolo_conf=0.20, min_score=0.20, **kwargs):
    """Seçilen sınıf için yüksek doğruluklu etiketleme motoru (Esnek ve Hata Korumalı)"""
    print(f"\n" + "="*60)
    print(f"📌 {category_name.upper()} SINIFI ETİKETLENİYOR (Class ID: {class_id})")
    print("="*60)

    # 1. Modelleri Hazırla
    ontology = CaptionOntology(dino_prompts)
    model_dino = GroundingDINO(ontology=ontology, box_threshold=box_threshold)
    model_yolo = YOLOWorld("yolov8x-worldv2.pt")
    model_yolo.set_classes(yolo_prompts)

    # 2. Klasör Yollarını Ayarla
    input_dir = os.path.join(BASE_DIR, category_name)
    if not os.path.exists(input_dir):
        input_dir = os.path.join("/content/gdrive/MyDrive", category_name)

    output_dir = os.path.join(os.path.dirname(input_dir), f"{category_name}_labeled")
    labels_dir = os.path.join(output_dir, "labels")
    previews_dir = os.path.join(output_dir, "previews")

    os.makedirs(labels_dir, exist_ok=True)
    os.makedirs(previews_dir, exist_ok=True)

    image_paths = sorted(
        glob.glob(os.path.join(input_dir, "*.[jJ][pP][gG]")) +
        glob.glob(os.path.join(input_dir, "*.[pP][nN][gG]")) +
        glob.glob(os.path.join(input_dir, "*.jpeg")) +
        glob.glob(os.path.join(input_dir, "*.webp"))
    )

    print(f"📂 Bulunan Toplam Görsel: {len(image_paths)}")
    success_count = 0
    skipped_count = 0

    for idx, img_path in enumerate(image_paths):
        filename = os.path.basename(img_path)
        base_name = os.path.splitext(filename)[0]
        txt_path = os.path.join(labels_dir, f"{base_name}.txt")

        # Resume: Zaten etiketlenmişse atla
        if os.path.exists(txt_path):
            skipped_count += 1
            continue

        try:
            pil_img = Image.open(img_path).convert("RGB")
            img_w, img_h = pil_img.size
            cv_img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
            candidate_boxes = []

            # A) Grounding DINO Tespiti
            try:
                dino_res = model_dino.predict(img_path)
                if dino_res is not None and len(dino_res.xyxy) > 0:
                    for b in dino_res.xyxy:
                        candidate_boxes.append(b)
            except Exception:
                pass

            # B) YOLO-World Tespiti
            try:
                yolo_res = model_yolo.predict(img_path, conf=yolo_conf, verbose=False)[0]
                if yolo_res.boxes is not None and len(yolo_res.boxes.xyxy) > 0:
                    for b in yolo_res.boxes.xyxy.cpu().numpy():
                        candidate_boxes.append(b)
            except Exception:
                pass

            valid_boxes = []
            yolo_txt_lines = []

            # C) SigLIP Doğrulama ve IoU Filtreleme
            for box in candidate_boxes:
                xmin, ymin, xmax, ymax = map(int, box)
                xmin, ymin = max(0, xmin), max(0, ymin)
                xmax, ymax = min(img_w, xmax), min(img_h, ymax)

                if (xmax - xmin) < 15 or (ymax - ymin) < 15:
                    continue

                crop = pil_img.crop((xmin, ymin, xmax, ymax))
                is_valid, score = verify_crop(crop, category_name, min_score=min_score)

                if is_valid:
                    # IoU Mükerrer Eleme
                    is_duplicate = False
                    for eb in valid_boxes:
                        ix1 = max(xmin, eb[0]); iy1 = max(ymin, eb[1])
                        ix2 = min(xmax, eb[2]); iy2 = min(ymax, eb[3])
                        inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
                        a1 = (xmax - xmin) * (ymax - ymin)
                        a2 = (eb[2] - eb[0]) * (eb[3] - eb[1])
                        union = a1 + a2 - inter
                        if union > 0 and (inter / union) > 0.50:
                            is_duplicate = True
                            break

                    if not is_duplicate:
                        valid_boxes.append([xmin, ymin, xmax, ymax])
                        xc = ((xmin + xmax) / 2.0) / img_w
                        yc = ((ymin + ymax) / 2.0) / img_h
                        bw = (xmax - xmin) / img_w
                        bh = (ymax - ymin) / img_h
                        yolo_txt_lines.append(f"{class_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")

            # YOLO Formatında .txt Kaydet
            with open(txt_path, "w") as f:
                f.writelines(yolo_txt_lines)

            # Önizleme Görseli Kaydet
            if len(valid_boxes) > 0:
                dets = sv.Detections(
                    xyxy=np.array(valid_boxes),
                    class_id=np.zeros(len(valid_boxes), dtype=int)
                )
                lbs = [category_name for _ in valid_boxes]
                ann = box_annotator.annotate(scene=cv_img.copy(), detections=dets)
                ann = label_annotator.annotate(scene=ann, detections=dets, labels=lbs)
                cv2.imwrite(os.path.join(previews_dir, f"{base_name}.jpg"), ann)
                success_count += 1
                print(f"✅ [{idx+1}/{len(image_paths)}] {filename} -> {len(valid_boxes)} adet {category_name} etiketlendi!")
            else:
                print(f"⚪ [{idx+1}/{len(image_paths)}] {filename} -> Tespit edilemedi (DINO/YOLO: {len(candidate_boxes)} aday)")

        except Exception as e:
            print(f"⚠️ Hata ({filename}): {e}")

    print(f"\n🎉 {category_name.upper()} Tamamlandı!")
    print(f"📊 Yeni Etiketlenen: {success_count} | Zaten Etiketlenmiş: {skipped_count} | Toplam: {len(image_paths)}\n")

## 👜 1. BAG (Çanta - Class ID: 0)

In [ ]:
run_auto_labeling(
    category_name="bag",
    class_id=0,
    dino_prompts={"bag": "bag", "handbag": "bag", "backpack": "bag", "suitcase": "bag", "tote bag": "bag", "shoulder bag": "bag", "clutch": "bag", "purse": "bag"},
    yolo_prompts=["bag", "handbag", "backpack", "suitcase", "purse", "tote bag"],
    box_threshold=0.20,
    yolo_conf=0.20,
    min_score=0.20
)

## 🎗️ 2. BELT (Kemer - Class ID: 1)

In [ ]:
run_auto_labeling(
    category_name="belt",
    class_id=1,
    dino_prompts={"belt": "belt", "leather belt": "belt", "waist belt": "belt", "strap belt": "belt"},
    yolo_prompts=["belt"],
    box_threshold=0.20,
    yolo_conf=0.20,
    min_score=0.20
)

## 🧤 3. GLOVES (Eldiven - Class ID: 2)

In [ ]:
run_auto_labeling(
    category_name="gloves",
    class_id=2,
    dino_prompts={"glove": "gloves", "gloves": "gloves", "pair of gloves": "gloves", "mittens": "gloves", "winter gloves": "gloves"},
    yolo_prompts=["gloves", "glove"],
    box_threshold=0.20,
    yolo_conf=0.20,
    min_score=0.20
)

## 📿 4. NECKLACE (Kolye - Class ID: 3)

In [ ]:
run_auto_labeling(
    category_name="necklace",
    class_id=3,
    dino_prompts={"necklace": "necklace", "pendant": "necklace", "chain necklace": "necklace", "jewelry necklace": "necklace", "choker": "necklace"},
    yolo_prompts=["necklace"],
    box_threshold=0.18,
    yolo_conf=0.18,
    min_score=0.18
)

## 🧣 5. SCARF (Atkı / Şal - Class ID: 4)

In [ ]:
run_auto_labeling(
    category_name="scarf",
    class_id=4,
    dino_prompts={"scarf": "scarf", "shawl": "scarf", "muffler": "scarf", "neck scarf": "scarf", "winter scarf": "scarf"},
    yolo_prompts=["scarf"],
    box_threshold=0.20,
    yolo_conf=0.20,
    min_score=0.20
)

## 👔 6. TIE (Kravat / Papyon - Class ID: 5)

In [ ]:
run_auto_labeling(
    category_name="tie",
    class_id=5,
    dino_prompts={"necktie": "tie", "tie": "tie", "bow tie": "tie", "suit tie": "tie"},
    yolo_prompts=["tie"],
    box_threshold=0.20,
    yolo_conf=0.20,
    min_score=0.20
)

## ⌚ 7. WATCH (Kol Saati - Class ID: 6)

In [ ]:
run_auto_labeling(
    category_name="watch",
    class_id=6,
    dino_prompts={"wristwatch": "watch", "watch": "watch", "smartwatch": "watch", "wrist watch": "watch"},
    yolo_prompts=["watch"],
    box_threshold=0.18,
    yolo_conf=0.18,
    min_score=0.20
)

## 🧦 8. SOCK (Çorap - Class ID: 7)

In [ ]:
import os, shutil

# Sıfırdan başlamak için klasörleri temizle
labels_dir = "/content/gdrive/MyDrive/kiyafetler/sock_labeled/labels"
previews_dir = "/content/gdrive/MyDrive/kiyafetler/sock_labeled/previews"

for path in [labels_dir, previews_dir]:
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)

print("🧹 Eski veriler temizlendi! 1. resimden başlanıyor...")

# Agresif algılama parametreleri ile çalıştırma
run_auto_labeling(
    category_name="sock",
    class_id=7,
    dino_prompts={
        "sock": "sock",
        "socks": "sock",
        "pair of socks": "sock",
        "ankle sock": "sock",
        "crew sock": "sock",
        "cotton socks": "sock",
        "stockings": "sock"
    },
    yolo_prompts=["sock", "socks", "stockings"],
    box_threshold=0.18,
    yolo_conf=0.18,
    min_score=0.18
)

## 📦 ADIM 5: Tüm Sınıfları Tek Bir YOLO Veri Setinde Birleştirme & Train/Val Split
Etiketlediğiniz tüm sınıfların resimlerini ve `.txt` etiketlerini `%80 Train` ve `%20 Val` olarak böler, otomatik `data.yaml` dosyasını oluşturur.

In [ ]:
import os, shutil, random, glob, yaml

BASE_DIR          = "/content/gdrive/MyDrive/kiyafetler"
DATASET_OUTPUT_DIR = "/content/gdrive/MyDrive/yolo_dataset"
CATEGORIES        = ["bag", "belt", "gloves", "necklace", "scarf", "tie", "watch"]

train_img_dir = os.path.join(DATASET_OUTPUT_DIR, "images", "train")
val_img_dir   = os.path.join(DATASET_OUTPUT_DIR, "images", "val")
train_lbl_dir = os.path.join(DATASET_OUTPUT_DIR, "labels", "train")
val_lbl_dir   = os.path.join(DATASET_OUTPUT_DIR, "labels", "val")

for d in [train_img_dir, val_img_dir, train_lbl_dir, val_lbl_dir]:
    os.makedirs(d, exist_ok=True)

total_copied = 0
print("🚀 Veri seti birleştiriliyor ve çakışmasız isimlendiriliyor...")

for cat in CATEGORIES:
    labeled_dir = os.path.join(BASE_DIR, f"{cat}_labeled")
    if not os.path.exists(labeled_dir):
        labeled_dir = os.path.join("/content/gdrive/MyDrive", f"{cat}_labeled")

    labels_folder     = os.path.join(labeled_dir, "labels")
    raw_images_folder = os.path.join(BASE_DIR, cat)
    if not os.path.exists(raw_images_folder):
        raw_images_folder = os.path.join("/content/gdrive/MyDrive", cat)

    if not os.path.exists(labels_folder):
        print(f"⚠️ {cat} etiketi bulunamadı, atlanıyor.")
        continue

    valid_pairs = []
    for txt_path in glob.glob(os.path.join(labels_folder, "*.txt")):
        base_name = os.path.splitext(os.path.basename(txt_path))[0]
        found_img = None
        for ext in [".jpg", ".jpeg", ".png", ".webp", ".JPG", ".PNG"]:
            candidate = os.path.join(raw_images_folder, base_name + ext)
            if os.path.exists(candidate):
                found_img = candidate
                break
        if found_img and os.path.getsize(txt_path) > 0:
            valid_pairs.append((found_img, txt_path))

    random.seed(42)
    random.shuffle(valid_pairs)
    split_idx   = int(len(valid_pairs) * 0.80)
    train_pairs = valid_pairs[:split_idx]
    val_pairs   = valid_pairs[split_idx:]

    # Train kopyalama (Sınıf ön ekiyle güvenli adlandırma)
    for img_p, lbl_p in train_pairs:
        base_fname = os.path.splitext(os.path.basename(img_p))[0]
        img_ext = os.path.splitext(img_p)[1].lower()
        target_base = f"{cat}_{base_fname}" if not base_fname.startswith(f"{cat}_") else base_fname
        shutil.copy2(img_p, os.path.join(train_img_dir, f"{target_base}{img_ext}"))
        shutil.copy2(lbl_p, os.path.join(train_lbl_dir, f"{target_base}.txt"))

    # Val kopyalama
    for img_p, lbl_p in val_pairs:
        base_fname = os.path.splitext(os.path.basename(img_p))[0]
        img_ext = os.path.splitext(img_p)[1].lower()
        target_base = f"{cat}_{base_fname}" if not base_fname.startswith(f"{cat}_") else base_fname
        shutil.copy2(img_p, os.path.join(val_img_dir, f"{target_base}{img_ext}"))
        shutil.copy2(lbl_p, os.path.join(val_lbl_dir, f"{target_base}.txt"))

    print(f"✅ {cat.upper()}: {len(train_pairs)} train | {len(val_pairs)} val")
    total_copied += len(valid_pairs)

data_yaml_dict = {
    "path": DATASET_OUTPUT_DIR,
    "train": "images/train",
    "val":   "images/val",
    "names": {0: "bag", 1: "belt", 2: "gloves", 3: "necklace",
              4: "scarf", 5: "tie", 6: "watch", 7: "sock"}
}

yaml_path = os.path.join(DATASET_OUTPUT_DIR, "data.yaml")
with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.dump(data_yaml_dict, f, default_flow_style=False, allow_unicode=True)

print(f"\n🎉 Veri seti hazır!")
print(f"📁 Konum: {DATASET_OUTPUT_DIR}")
print(f"📄 data.yaml: {yaml_path}")
print(f"📊 Toplam etiketli resim: {total_copied}")

## 🗜️ ADIM 6: Veri Setini ZIP Olarak Paketleme (Opsiyonel)

In [ ]:
import shutil
print("📦 Veri seti zip dosyasına sıkıştırılıyor...")
shutil.make_archive("/content/gdrive/MyDrive/yolo_dataset", 'zip', "/content/gdrive/MyDrive/yolo_dataset")
print("✅ Tamamlandı: /content/gdrive/MyDrive/yolo_dataset.zip hazır!")